In [5]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
username = "aacuser"
password = "abc123"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)


image_filename = '../code_files/Grazioso Salvare Logo.png' 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),

    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '150px'}
            ),
            href='https://www.snhu.edu'
        )
    ),

    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),

    html.Center(html.H3('Created by Ken Custer')),

    html.Hr(),
            

    html.Div([
    html.Label('Filter by Rescue Type:'),

    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Water Rescue', 'value': 'Water'},
            {'label': 'Mountain or Wilderness Rescue', 'value': 'Mountain'},
            {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
            {'label': 'Reset', 'value': 'Reset'}
        ],
        value='Reset',
        labelStyle={'display': 'inline-block', 'margin-right': '15px'}
    )
]),
# Show how many animals match the current filter.
html.Div(
    id='record-count',
    style={
        'fontWeight': 'bold',
        'margin': '10px 0'
    }
),   
    
dash_table.DataTable(
    id='datatable-id',
    columns=[
        {"name": i, "id": i, "deletable": False, "selectable": True}
        for i in df.columns
    ],
    data=df.to_dict('records'),

    # Allow the user to select one animal for the map.
    row_selectable='single',
    selected_rows=[0],

    # Show 10 records at a time.
    page_action='native',
    page_current=0,
    page_size=10,

    # Allow the user to sort the table.
    sort_action='native',
    
    # Allow the current table data to be downloaded.
    export_format='csv',
    export_headers='display',
    
    # Keep the table readable when there are many columns.
    style_table={
        'overflowX': 'auto'
    },

    style_cell={
        'textAlign': 'left',
        'minWidth': '100px',
        'maxWidth': '180px',
        'whiteSpace': 'normal'
    },

    style_header={
        'fontWeight': 'bold'
    }
    
),
    html.Br(),
    html.Hr(),

    # Place the pie chart and map next to each other.
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),

            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )

])

#############################################
# Interaction Between Components / Controller
#############################################
# Store the MongoDB queries for each rescue type.
rescue_queries = {
    'Water': {
        'animal_type': 'Dog',
        'breed': {'$in': [
            'Labrador Retriever Mix',
            'Chesapeake Bay Retriever',
            'Newfoundland'
        ]},
        'sex_upon_outcome': 'Intact Female',
        'age_upon_outcome_in_weeks': {
            '$gte': 26,
            '$lte': 156
        }
    },

    'Mountain': {
        'animal_type': 'Dog',
        'breed': {'$in': [
            'German Shepherd',
            'Alaskan Malamute',
            'Old English Sheepdog',
            'Siberian Husky',
            'Rottweiler'
        ]},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {
            '$gte': 26,
            '$lte': 156
        }
    },

    'Disaster': {
        'animal_type': 'Dog',
        'breed': {'$in': [
            'Doberman Pinscher',
            'German Shepherd',
            'Golden Retriever',
            'Bloodhound',
            'Rottweiler'
        ]},
        'sex_upon_outcome': 'Intact Male',
        'age_upon_outcome_in_weeks': {
            '$gte': 20,
            '$lte': 300
        }
    }
}
    
@app.callback(Output('datatable-id', 'data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):

    # Get the query for the selected rescue type.
    # Reset uses an empty query to return all records.
    query = rescue_queries.get(filter_type, {})

    # Retrieve the matching records from MongoDB.
    dff = pd.DataFrame.from_records(db.read(query))

    # Remove MongoDB's ObjectId before sending data to the table.
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Update the number of animals shown in the table.
@app.callback(
    Output('record-count', 'children'),
    [Input('datatable-id', 'data')]
)
def update_record_count(data):

    if data is None:
        count = 0
    else:
        count = len(data)

    return f'Animals Shown: {count}'


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    # Use the data currently shown in the table.
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Do not create a chart if there are no records.
    if dff.empty:
        return []

    # Count the breeds and keep the 10 most common.
    breed_counts = dff['breed'].value_counts().head(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names='breed',
                values='count',
                title='Top 10 Animal Breeds'
            )
        )
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    # Nothing is highlighted until a column is selected.
    if not selected_columns:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')])
def update_map(viewData, index):

    # Use the data currently displayed in the table.
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # If there are no records, return an empty map.
    if dff.empty:
        return [
            dl.Map(
                style={'width': '100%', 'height': '500px'},
                center=[30.75, -97.48],
                zoom=10,
                children=[
                    dl.TileLayer(id='base-layer-id')
                ]
            )
        ]

    # Use the first row if the user has not selected one.
    if index is None or len(index) == 0 or index[0] >= len(dff):
        row = 0
    else:
        row = index[0]

    # Get the selected animal's location.
    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[latitude, longitude],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[latitude, longitude],
                    children=[
                        dl.Tooltip(dff.iloc[row]['breed']),
                        dl.Popup([
                            html.H3(dff.iloc[row]['name']),
                            html.P(
                                'Breed: ' +
                                str(dff.iloc[row]['breed'])
                            ),
                            html.P(
                                'Sex: ' +
                                str(dff.iloc[row]['sex_upon_outcome'])
                            ),
                            html.P(
                                'Age: ' +
                                str(round(
                                    dff.iloc[row]['age_upon_outcome_in_weeks'], 1
                                )) +
                                ' weeks'
                            )
                        ])
                    ]
                )
            ]
        )
    ]
    
# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://staranagram-visitoralarm-3000.codio.io/proxy/8050/
